<a href="https://colab.research.google.com/github/jim08079/Vision-AI-V2/blob/main/AI_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, glob, json, shutil, random
from PIL import Image

# 1. 解壓縮資料集
!unzip -q /content/my_yolo_dataset.zip -d /content/raw_data

# 自動尋找 classes.txt 所在的資料夾
base_dir = "/content/raw_data/my_yolo_dataset"
if not os.path.exists(os.path.join(base_dir, "classes.txt")):
    base_dir = "/content/raw_data"

with open(os.path.join(base_dir, "classes.txt"), "r") as f:
    classes = [line.strip() for line in f.readlines() if line.strip()]
print(f"✅ 成功讀取 {len(classes)} 個類別: {classes}")
print(f"⚠️ 請記下你的類別總數 num_classes = {len(classes)}，以及上方的類別順序！")

# 2. 建立 COCO 資料夾並切分 80% 訓練集 / 20% 驗證集
coco_root = "/content/coco_dataset"
for split in ["train", "valid"]:
    os.makedirs(os.path.join(coco_root, split), exist_ok=True)

img_files = glob.glob(os.path.join(base_dir, "images", "*.*"))
random.shuffle(img_files)
split_idx = int(len(img_files) * 0.8)
splits = {"train": img_files[:split_idx], "valid": img_files[split_idx:]}

for split, files in splits.items():
    coco_dict = {
        "images": [],
        "annotations": [],
        "categories": [{"id": i, "name": name, "supercategory": "none"} for i, name in enumerate(classes)]
    }
    ann_id = 1
    for img_id, img_path in enumerate(files, 1):
        filename = os.path.basename(img_path)
        shutil.copy(img_path, os.path.join(coco_root, split, filename))
        with Image.open(img_path) as img:
            w, h = img.size
        coco_dict["images"].append({"id": img_id, "file_name": filename, "width": w, "height": h})

        txt_name = os.path.splitext(filename)[0] + ".txt"
        txt_path = os.path.join(base_dir, "labels", txt_name)
        if os.path.exists(txt_path):
            with open(txt_path, "r") as lf:
                for line in lf:
                    parts = line.strip().split()
                    if len(parts) != 5: continue
                    cls_id = int(parts[0])
                    cx, cy, bw, bh = map(float, parts[1:])
                    xmin = (cx - bw / 2) * w
                    ymin = (cy - bh / 2) * h
                    abs_w = bw * w
                    abs_h = bh * h
                    coco_dict["annotations"].append({
                        "id": ann_id, "image_id": img_id, "category_id": cls_id,
                        "bbox": [xmin, ymin, abs_w, abs_h], "area": abs_w * abs_h,
                        "iscrowd": 0, "segmentation": []
                    })
                    ann_id += 1

    with open(os.path.join(coco_root, split, "_annotations.coco.json"), "w") as out_f:
        json.dump(coco_dict, out_f)
print("✅ COCO 格式轉換完成！")

✅ 成功讀取 6 個類別: ['Ball', 'mose', 'box', 'body', 'water', 'trash']
⚠️ 請記下你的類別總數 num_classes = 6，以及上方的類別順序！
✅ COCO 格式轉換完成！


In [ ]:
%cd /content
!git clone https://github.com/Seeed-Studio/ModelAssistant.git -b main
%cd /content/ModelAssistant
!pip install -r requirements/base.txt -r requirements/export.txt -r requirements/albu.txt
!pip install -e .
!pip install ethos-u-vela

/content
Cloning into 'ModelAssistant'...
remote: Enumerating objects: 15958, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 15958 (delta 44), reused 27 (delta 27), pack-reused 15885 (from 2)
Receiving objects: 100% (15958/15958), 26.77 MiB | 19.37 MiB/s, done.
Resolving deltas: 100% (9262/9262), done.
/content/ModelAssistant
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements/base.txt'
Obtaining file:///content/ModelAssistant
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 5.3 MB/s eta 0:00:00
ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_lo

In [ ]:
%cd /content/ModelAssistant
!pip install ethos-u-vela==3.9.0 tensorflow==2.12.0
!bash scripts/setup_colab.sh
# 如果找不到 setup 腳本，請手動執行： !pip install -e .

/content/ModelAssistant
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.5/386.5 kB 24.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Could not find a version that satisfies the requirement tensorflow==2.12.0 (from versions: 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0, 2.22.0rc0)
ERROR: No matching distribution found for tensorflow==2.12.0
Checking if CUDA available... OK
Ensuring PyTorch stack... 
torch=2.11.0+cu130 torchvision=0.26.0+cu130 torchaudio=2.11.0+cu130
torch==2.11.0+cu130
torchvision==0.26.0+cu130
torchaudio==2.11.0+cu130
Installing build tools... 
Installing dependencies... 
Installing sscma... 
Obtaining file:///content/ModelAssistant
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editab

In [ ]:
!pip install tensorflow

In [ ]:
%%writefile vela_config.ini
[System_Config.Ethos_U55_High_End_Embedded]
core_clock=400e6
axi0_port=Sram
axi1_port=OffChipFlash
Sram_clock_scale=1.0
Sram_burst_length=32
Sram_read_latency=16
Sram_write_latency=16
Dram_clock_scale=0.75
Dram_burst_length=128
Dram_read_latency=500
Dram_write_latency=250
OnChipFlash_clock_scale=0.25
OffChipFlash_clock_scale=0.015625
OffChipFlash_burst_length=32
OffChipFlash_read_latency=64
OffChipFlash_write_latency=64

[Memory_Mode.Shared_Sram]
const_mem_area=Axi1
arena_mem_area=Axi0
cache_mem_area=Axi0

Writing vela_config.ini


In [ ]:
%cd /content/ModelAssistant

# 1. 訓練 Swift-YOLO 多目標模型
!python tools/train.py configs/swift_yolo/swift_yolo_tiny_1xb16_300e_coco.py \
    --cfg-options \
    data_root='/content/coco_dataset/' \
    train_ann_file='train/_annotations.coco.json' \
    val_ann_file='valid/_annotations.coco.json' \
    train_data_prefix='train/' \
    val_data_prefix='valid/' \
    num_classes=6 \
    epochs=100 \
    imgsz="(192, 192)"

/content/ModelAssistant
10/20 07:24:03 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 507746217
    GPU 0: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 13.0, V13.0.88
    GCC: x86_64-linux-gnu-gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
    PyTorch: 2.11.0+cu130
    PyTorch compiling details: PyTorch built with:
  - GCC 13.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2024.2-Product Build 20240605 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.10.2 (Git Hash f1d471933dc852f956fd05389f9313c7148783d5)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 13.0
  - NVCC architecture flags: -ge

In [ ]:
%cd /content/ModelAssistant

# 2. 將訓練好的權重匯出為 INT8 TFLite (修改校準集為 valid)
!python tools/export.py configs/swift_yolo/swift_yolo_tiny_1xb16_300e_coco.py \
    work_dirs/swift_yolo_tiny_1xb16_300e_coco/epoch_100.pth \
    --format tflite \
    --image_path '/content/coco_dataset/valid/' \
    --cfg-options \
    data_root='/content/coco_dataset/' \
    train_ann_file='train/_annotations.coco.json' \
    val_ann_file='valid/_annotations.coco.json' \
    train_data_prefix='train/' \
    val_data_prefix='valid/' \
    num_classes=6 \
    imgsz="(192, 192)"

/content/ModelAssistant
10/20 08:12:12 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 2076876547
    GPU 0: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 13.0, V13.0.88
    GCC: x86_64-linux-gnu-gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
    PyTorch: 2.11.0+cu130
    PyTorch compiling details: PyTorch built with:
  - GCC 13.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2024.2-Product Build 20240605 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.10.2 (Git Hash f1d471933dc852f956fd05389f9313c7148783d5)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 13.0
  - NVCC architecture flags: -g

In [ ]:
%cd /content/ModelAssistant

# 3. 使用 Vela 編譯新的硬體專用格式
!vela work_dirs/swift_yolo_tiny_1xb16_300e_coco/epoch_100_int8.tflite \
    --accelerator-config ethos-u55-64 \
    --config vela_config.ini \
    --system-config Ethos_U55_High_End_Embedded \
    --memory-mode Shared_Sram \
    --output-dir ./vela_output

/content/ModelAssistant
Error: Address out of bounds, address 16384, memory 'shram' with size 16384
Error: Address offset out of bounds, address 16384, offset 256, memory 'shram' with size 16384
Error: Address out of bounds, address 16384, memory 'shram' with size 16384
Error: Address offset out of bounds, address 16384, offset 256, memory 'shram' with size 16384
Error: Address out of bounds, address 16384, memory 'shram' with size 16384
Error: Address offset out of bounds, address 16384, offset 256, memory 'shram' with size 16384

Network summary for epoch_100_int8
Accelerator configuration                Ethos_U55_64
System configuration             Ethos_U55_High_End_Embedded
Memory mode                               Shared_Sram
Accelerator clock                                 400 MHz
Design peak SRAM bandwidth                       2.98 GB/s
Design peak Off-chip Flash bandwidth             0.05 GB/s

Total SRAM used                                705.83 KiB
Total Off-chip Flash us